In [165]:
import pandas as pd

In [166]:
df = pd.read_csv('../data/challenge_381.csv')

df.head()

,Employee ID,Employee Name,January,February,March,April,May,June,July,August,September,October,November,December
0,1,Barbara Somers,0,8,3,3,0,5,3,6,1,7,0,6
1,2,Benjamin Portal,2,6,2,3,4,4,3,3,1,5,4,5
2,3,Beth Silverstein,0,7,7,2,0,8,6,0,9,0,3,0
3,4,Carinne Cardona,2,4,1,0,3,3,5,1,9,7,1,6
4,5,Chahait Singh,0,0,12,0,0,0,10,9,0,3,0,2


In [167]:
# unpivot df

df = pd.melt(df, id_vars=['Employee ID', 'Employee Name'])

df.head()

,Employee ID,Employee Name,variable,value
0,1,Barbara Somers,January,0
1,2,Benjamin Portal,January,2
2,3,Beth Silverstein,January,0
3,4,Carinne Cardona,January,2
4,5,Chahait Singh,January,0


In [168]:
# convert month names to dates for sorting

df['date'] = pd.to_datetime('2000-' + df['variable'] + '-01', format='%Y-%B-%d')
df = df.sort_values(by=['Employee Name', 'date'], ascending=True)

df.head()

,Employee ID,Employee Name,variable,value,date
0,1,Barbara Somers,January,0,2000-01-01
50,1,Barbara Somers,February,8,2000-02-01
100,1,Barbara Somers,March,3,2000-03-01
150,1,Barbara Somers,April,3,2000-04-01
200,1,Barbara Somers,May,0,2000-05-01


In [169]:
# create a new df with only the first non-zero values

df_2 = df[df['value'] != 0]
idx = df_2.groupby('Employee ID')['date'].idxmin()
df_2 = df_2.loc[idx][['Employee ID', 'date']]
df_2 = df_2.rename(columns={'date': 'first_month'})

df_2.head()

,Employee ID,first_month
50,1,2000-02-01
1,2,2000-01-01
52,3,2000-02-01
3,4,2000-01-01
104,5,2000-03-01


In [170]:
# append first month to each row for comparison

df = pd.merge(df, df_2, how='outer', on='Employee ID')\

# filter to only the first month and the two after

df = df[(df['date'] >= df['first_month'])
        & (df['date'] <= df['first_month'] + pd.offsets.DateOffset(months=2))]

df.head()

,Employee ID,Employee Name,variable,value,date,first_month
1,1,Barbara Somers,February,8,2000-02-01,2000-02-01
2,1,Barbara Somers,March,3,2000-03-01,2000-02-01
3,1,Barbara Somers,April,3,2000-04-01,2000-02-01
12,2,Benjamin Portal,January,2,2000-01-01,2000-01-01
13,2,Benjamin Portal,February,6,2000-02-01,2000-01-01


In [171]:
# add up the sales for each employee

df = df.groupby(['Employee ID', 'Employee Name'])['value'].sum().reset_index()

# calculate the three month average

df['average sales'] = round(df['value'] / 3, 2)

df.head()

,Employee ID,Employee Name,value,average sales
0,1,Barbara Somers,14,4.67
1,2,Benjamin Portal,10,3.33
2,3,Beth Silverstein,16,5.33
3,4,Carinne Cardona,7,2.33
4,5,Chahait Singh,12,4.00
